# Basket option implementation with Bachelier model CV

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import numpy as np
import pyfeng as pf

#import option_models as opt
from option_models import basket

# How to reproduce RNs?
## Store the random state and restore

In [14]:
rng = np.random.default_rng()

In [20]:
#state = rng.bit_generator.state
#print(state)
rng.standard_normal((4,4))

{'bit_generator': 'PCG64', 'state': {'state': 336716712067913489535001112599386548200, 'inc': 330695271609687404730686294007952799695}, 'has_uint32': 0, 'uinteger': 0}


array([[-0.9985911 , -2.09635952, -0.01484617,  1.63128583],
       [ 1.01900654,  0.29788118,  0.35883333, -1.24372447],
       [ 0.35527418, -0.08619562,  0.77865998,  0.52463935],
       [-0.72909757, -2.1670847 , -1.14304842, -0.81212897]])

In [21]:
rng.standard_normal((4,4))

array([[-0.89208069, -0.50419223,  1.47709873, -0.60294748],
       [-0.32134306, -0.47453045, -1.64684049, -1.53870242],
       [-1.08630333, -0.82853124, -2.02204806, -1.23137549],
       [-0.91352989,  0.75798009, -0.37450247,  0.63435883]])

In [22]:
rng.bit_generator.state = state

In [23]:
rng.standard_normal((4,4))

array([[-0.9985911 , -2.09635952, -0.01484617,  1.63128583],
       [ 1.01900654,  0.29788118,  0.35883333, -1.24372447],
       [ 0.35527418, -0.08619562,  0.77865998,  0.52463935],
       [-0.72909757, -2.1670847 , -1.14304842, -0.81212897]])

# Basket option on trivial cases

In [26]:
# A trivial test case 1: 
# one asset have 100% weight (the others zero)
# the case should be equivalent to the BSM or Normal model price

spot = np.ones(4) * 100
vol = np.ones(4) * 0.4
weights = np.array([1, 0, 0, 0])
divr = np.zeros(4)
intr = 0
cor_m = 0.5*np.identity(4) + 0.5
texp = 5
strike = 120

In [27]:
cor_m

array([[1. , 0.5, 0.5, 0.5],
       [0.5, 1. , 0.5, 0.5],
       [0.5, 0.5, 1. , 0.5],
       [0.5, 0.5, 0.5, 1. ]])

In [30]:
print(weights)

price_basket = basket.basket_price_mc(strike, spot, vol*spot, weights, texp, cor_m, bsm=False)
print(price_basket)

[1 0 0 0]
26.461091839489136


In [31]:
# Compare the price to normal model formula

norm1 = pf.Norm(sigma=40)
price_norm = norm1.price(strike, spot[0], texp, cp=1)
print(price_basket, price_norm)

26.461091839489136 26.570845957870507


In [32]:
# A trivial test case 2
# all assets almost perfectly correlated:
# the case should be equivalent to the BSM or Normal model price

spot = np.ones(4) * 100
vol = np.ones(4) * 0.4
weights = np.ones(4) * 0.25
divr = np.zeros(4)
intr = 0
cor_m = 0.0001*np.identity(4) + 0.9999*np.ones((4,4))
texp = 5
strike = 120

print( cor_m )

price_basket = basket.basket_price_mc(strike, spot, vol*spot, weights, texp, cor_m, bsm=False)
print(price_basket, price_norm)

[[1.     0.9999 0.9999 0.9999]
 [0.9999 1.     0.9999 0.9999]
 [0.9999 0.9999 1.     0.9999]
 [0.9999 0.9999 0.9999 1.    ]]
26.680830816103004 26.570845957870507


In [33]:
# A full test set for basket option with exact price

spot = np.ones(4) * 100
vol = np.ones(4) * 0.4
weights = np.ones(4) * 0.25
divr = np.zeros(4)
intr = 0
cor_m = 0.5*np.identity(4) + 0.5
texp = 5
strike = 100
price_exact = 28.0073695

In [34]:
weights, cor_m

(array([0.25, 0.25, 0.25, 0.25]),
 array([[1. , 0.5, 0.5, 0.5],
        [0.5, 1. , 0.5, 0.5],
        [0.5, 0.5, 1. , 0.5],
        [0.5, 0.5, 0.5, 1. ]]))

In [35]:
price_basket = basket.basket_price_mc(strike, spot, vol*spot, weights, texp, cor_m, bsm=False)
print(price_basket, price_exact)

28.121133197337674 28.0073695


# [To Do] Basket option implementation based on BSM model
## Write the similar test for BSM

In [36]:
price_basket = basket.basket_price_mc(strike, spot, vol, weights, texp, cor_m, bsm=True)
print(price_basket)

0.0


In [37]:
# A trivial test case 1: 
# one asset have 100% weight (the others zero)
# the case should be equivalent to the BSM or Normal model price

spot = np.ones(4) * 100
vol = np.ones(4) * 0.4
weights = np.array([1, 0, 0, 0])
divr = np.zeros(4)
intr = 0
cor_m = 0.5*np.identity(4) + 0.5
texp = 5
strike = 120

print(weights)

np.random.seed(123456)
price_basket = basket.basket_price_mc(strike, spot, vol, weights, texp, cor_m, bsm=True)

[1 0 0 0]


In [38]:
bsm1 = pf.Bsm(sigma=vol[0])
price_bsm = bsm1.price(strike, spot[0], texp, cp=1)
print(price_basket, price_bsm)

0.0 28.713486748445934


# Spread option implementation based on normal model

In [14]:
# A full test set for spread option

spot = np.array([100, 96])
vol = np.array([0.2, 0.1])
weights = np.array([1, -1])
divr = np.array([1, 1])*0.05
intr = 0.1
cor = 0.5
cor_m = np.array([[1, cor], [cor, 1]])
texp = 1
strike = 0
price_exact = 8.5132252

In [15]:
# MC price based on normal model
# make sure that the prices are similar

np.random.seed(123456)
price_spread = basket.basket_price_mc(strike, spot, vol*spot, weights, texp, cor_m, intr=intr, divr=divr, bsm=False)
print(price_spread, price_exact)

8.317680907159142 8.5132252


# Spread option implementation based on BSM model

In [16]:
# Once the implementation is finished the BSM model price should also work
price_spread = basket.basket_price_mc(
    strike, spot, vol*spot, weights, texp, cor_m, intr=intr, divr=divr, bsm=True)
price_spread

0.0

In [17]:
# You also test Kirk's approximation
kirk = pf.BsmSpreadKirk(vol, cor=cor, divr=divr, intr=intr)
price_kirk = kirk.price(strike, spot, texp)
print(price_kirk, price_spread)

8.513225229545505 0.0


# [To Do] Complete the implementation of basket_price_norm_analytic
# Compare the MC stdev of BSM basket prices from with and without CV

In [39]:
# The basket option example from above
spot = np.ones(4) * 100
vol = np.ones(4) * 0.4
weights = np.array([1, 1, 1, 1])/4
divr = np.zeros(4)
intr = 0
cor_m = 0.5*np.identity(4) + 0.5
texp = 5
strike = 120

In [40]:
### Make sure that the analytic normal price is correctly implemented
basket.basket_price_norm_analytic(strike, spot, vol*spot, weights, texp, cor_m, intr=intr, divr=divr)

0.0

In [41]:
# Run below about 100 times and get the mean and stdev

### Returns 2 prices, without CV and with CV 
price_basket = basket.basket_price_mc_cv(strike, spot, vol, weights, texp, cor_m)

In [42]:
print(price_basket)

[0. 0.]


# Compare the MC variance

In [45]:
rng = np.random.default_rng()

vals = np.zeros((100, 2))

for i in np.arange(100):
    vals[i,:] = basket.basket_price_mc_cv(strike, spot, vol, weights, texp, cor_m, rng=rng)

In [48]:
np.mean(vals, axis=0), np.std(vals, axis=0)

(array([0., 0.]), array([0., 0.]))